# Gomoku MCTS Pytorch

Author xiaodongguaAIGC

五子棋 MCTS算法实现。代码实现follow AlphaGo-Zero

- agent带policy/value 网络
- 实现了state、node管理
- 实现了从零对弈
- 实现了policy/value损失
- 实现了mcts推理
- 阐述了LLM与MCTS的gap


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random
import copy

## config

In [2]:
board_size = 8
channel = 64
gomoku_number = 5  # 多少子连成一条线就算赢
exapand_size = 10
print(f'走子动作集合为:{board_size*board_size}')

走子动作集合为:64

## Gomoku Policy & Value Net Work

In [3]:
class GomokuNet(nn.Module):
    def __init__(self, board_size=15, channel=64):
        super(GomokuNet, self).__init__()
        self.board_size = board_size
        self.channel = channel
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, channel, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(channel * board_size * board_size, channel)
        self.fc2 = nn.Linear(channel, board_size * board_size)
        self.fc3 = nn.Linear(channel, 1)

    def forward(self, x):
        x1 = torch.relu(self.conv1(x))
        x2 = torch.relu(self.conv2(x1))
        x3 = x2.view(-1, self.channel * self.board_size * self.board_size)
        x4 = torch.relu(self.fc1(x3))
        policy = self.fc2(x4)
        value = torch.tanh(self.fc3(x4))
        return policy, value


model = GomokuNet(board_size=board_size, channel=channel)
# data = torch.zeros((1, 1, board_size, board_size), dtype=torch.float32)
# data = torch.randint(high = 2,size=(1, 1, board_size, board_size), dtype=torch.float32)
data = torch.randint(high=2, size=(
    1, 1, board_size, board_size), dtype=torch.float32)
policy, value = model(data)
print(policy.shape)
print(value.shape)

loss = policy[0].mean()
loss.backward()

torch.Size([1, 64])

torch.Size([1, 1])

## Mento Carlo Tree Searching

MCTS state

In [4]:
# 盘面数据
class GomokuState:
    def __init__(self, board_size=15, gomoku_number=4):
        self.board_size = board_size
        self.gomoku_number = gomoku_number

        # 盘面数据里每个格子的数据只有0(空)，1(我方)， 0(对方)
        self.board = torch.zeros(
            (1, 1, board_size, board_size), dtype=torch.float32)
        self.current_player = 1
        self.last_move = None

    def get_legal_actions(self):
        return torch.nonzero(self.board.view(-1) == 0).view(-1)
        # return torch.nonzero(self.board_int.view(-1) == 0)

    def is_terminal(self):
        # gomoku_number = 3
        if self.last_move is None:
            return False
        x, y = self.last_move
        player = self.board[0, 0, x, y]
        directions = [(1, 0), (0, 1), (1, 1), (1, -1)]
        for dx, dy in directions:
            count = 1
            for i in range(1, self.gomoku_number):
                nx, ny = x + i*dx, y + i*dy
                if 0 <= nx < self.board_size and 0 <= ny < self.board_size and self.board[0, 0, nx, ny] == player:
                    count += 1
                else:
                    break
            for i in range(1, gomoku_number):
                nx, ny = x - i*dx, y - i*dy
                if 0 <= nx < self.board_size and 0 <= ny < self.board_size and self.board[0, 0, nx, ny] == player:
                    count += 1
                else:
                    break
            if count >= self.gomoku_number:
                return True
        return len(self.get_legal_actions()) == 0

    def get_reward(self):
        if self.is_terminal():
            if self.current_player == -1:
                return 1  # Previous player (1) won
            else:
                return -1  # Previous player (-1) won
        return 0  # Game not finished

    # 对于盘面，是来回下子的，我方下子为1，对方下子为-1
    def move(self, action):
        x, y = action
        # 一定要clone，不然这里会变成in-place操作
        # 比如 t时刻 board^(t)， t时刻走子 board[0,0,x,y]=1
        # 那么在t时刻的board的数据就被替换了，将导致无法backward
        self.board = self.board.clone()
        # self.board[0, 0, x, y] = torch.tensor(self.current_player)
        # self.current_player = -torch.tensor(self.current_player)
        self.board[0, 0, x, y] = self.current_player
        self.current_player = -self.current_player
        self.last_move = action

    def clone(self):
        new_state = GomokuState(self.board_size)
        new_state.board = self.board.clone()
        new_state.current_player = self.current_player
        new_state.last_move = self.last_move
        return new_state


state = GomokuState(board_size=board_size, gomoku_number=gomoku_number)
print(f'可走子的策略为:{len(state.get_legal_actions())}')
# print(f'可走子的策略为:{state.get_legal_actions()}')

# 走子
state.move([0, 0])  # 我方
state.move([4, 0])  # 对手
state.move([0, 1])
state.move([4, 1])
print(f'是否终止:{state.is_terminal()}')
print(f'奖励:{state.get_reward()}')

state.move([0, 2])
state.move([4, 2])
state.move([0, 3])
state.move([4, 3])
state.move([0, 4])
# state.move([10,4])
print(f'是否终止:{state.is_terminal()}')
print(f'奖励:{state.get_reward()}')

可走子的策略为:64

是否终止:False

奖励:0

是否终止:True

奖励:1

In [5]:
# state = GomokuState(board_size=board_size)
print(f'可走子的策略为:{len(state.get_legal_actions())}')
print(f'可走子的策略为:{state.get_legal_actions()}')

可走子的策略为:55

可走子的策略为:tensor([ 5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
        23, 24, 25, 26, 27, 28, 29, 30, 31, 36, 37, 38, 39, 40, 41, 42, 43, 44,
        45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62,
        63])

### MCTS Node

In [6]:
class MCTSNode:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = {}
        self.visits = 0
        self.value = 0
        self.prior = 0

    def is_fully_expanded(self):
        return len(self.children) == len(self.state.get_legal_actions())

    def is_part_expanded(self):
        return len(self.children) == 10

    def select_child(self):
        return max(self.children.items(), key=lambda x: x[1].uct_value())

    def expand(self, policy):
        # 一次拓展
        valid_actions = self.state.get_legal_actions()
        policy_mask = policy[0, valid_actions]
        
        max_value, max_index = torch.max(policy_mask, dim=0)
        
        id = valid_actions[max_index].item()
        action = [int(id/self.state.board_size),
                  int(id % self.state.board_size)]

        # action = [3,3]
        # id = 12

        child_state = self.state.clone()
        child_state.board.detach()
        child_state.move(action)
        child_node = MCTSNode(child_state, self)
        child_node.prior = policy[0, id]         
        self.children[tuple(action)] = child_node
        return child_node

    def backpropagate(self, value):
        self.visits = self.visits + 1
        self.value = self.value + value
        if self.parent:
            self.parent.backpropagate(-value)

    def uct_value(self, c=1.4):
        if self.visits == 0:
            return float('inf')
        q = self.value / self.visits
        u = c * self.prior * math.sqrt(self.parent.visits) / (1 + self.visits)
        return q + u


state = GomokuState(board_size=board_size)
# 走子
state.move([0, 0])  # 我方
state.move([4, 0])  # 对手
state.move([0, 1])
state.move([4, 1])

node = MCTSNode(state)
policy, value = model(state.board)
print(policy.shape)
# policy2d = policy[0].view(board_size, board_size)
node.expand(policy)
node.backpropagate(2)
loss = (policy**2).mean()
loss.backward()
print(node.state.board)

torch.Size([1, 64])

tensor([[[[ 1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [-1., -1.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]]]])

In [7]:
print(node)
print(node.children)
# print(node.children[(3,2)])

<__main__.MCTSNode object at 0x126ca4bd0>

{(3, 4): <__main__.MCTSNode object at 0x10eaa4bd0>}

### MCTS Move

In [8]:
def mcts_move(state, net, num_simulations=1000):
    root = MCTSNode(state)
    for i in range(num_simulations):
        node = root

        # selection use UCB
        while len(node.children) != 0:
            node = node.select_child()[1] 
              
        # expansion
        if len(node.state.get_legal_actions()) != 0 and not node.state.is_terminal():
            policy, _ = net(node.state.board)
            policy = torch.softmax(policy, dim=1,) 
            node = node.expand(policy)   
                
        value = node.state.get_reward()
        if value == 0:  # If the game is not finished, use the neural network's evaluation
            _, value = net(node.state.board)
            value = value.item()  # 估计谁能赢

        # backup
        node.backpropagate(value)

    return max(root.children.items(), key=lambda x: x[1].visits)[0] # 实际选择执行的动作


a = mcts_move(state, model, num_simulations=10)
state.move(a)
final_reward = state.get_reward()
policy, _ = model(state.board)
loss = (policy**2).mean()*final_reward
loss.backward()
print(node.state.board)

tensor([[[[ 1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.],
          [-1., -1.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]]]])

## MCTS Training

In [12]:
# torch.autograd.set_detect_anomaly(True)
# 初始化神经网络
net = GomokuNet(board_size=board_size, channel=channel)
optimizer = optim.Adam(net.parameters(), lr=0.001)

episodes = 200
mcts_simulations = 1000

# 训练循环
for episode in range(episodes):  # 这个循环是用于训练 policy/value network
    state = GomokuState(board_size=board_size,
                        gomoku_number=gomoku_number)  
    states, policies, values, actions = [], [], [], []

    # t1:
    # 做MCTS 100次
    # 选出一个下子步骤
    # t2:
    # 做MCTS 100次
    # 选出一个下子步骤
    while not state.is_terminal():
        # if True:
        policy, value = net(state.board)  # 采样策略和价值估计
        policy = torch.softmax(policy, dim=1)

        # 模拟盘数
        action = mcts_move(state, net, mcts_simulations)  # mcts拓展 # 100次MCTS，都建在一棵树里，这棵树更新Q-Value。

        states.append(state.board)
        policies.append(policy)
        values.append(value)
        actions.append(action[0]*board_size + action[1])

        state.move(action)  # 执行下棋 take action

    # 计算真实的rewards
    final_reward = state.get_reward()

    target_values = torch.tensor(
        [final_reward * ((-1) ** i) for i in range(len(values))])
    # print(target_values.shape)

    # 训练网络
    optimizer.zero_grad()

    # Policy loss with CrossEntropy
    # pred  = [action x policys]
    # label = [action]
    pred = torch.cat(policies, dim=0)
    label = torch.tensor(actions)
    loss_fn = nn.CrossEntropyLoss()
    policy_loss = loss_fn(pred, label)
    
    # MSE
    value_loss = torch.mean((torch.cat(values) - target_values.detach()) ** 2)
    loss = policy_loss + value_loss
    print(policy_loss)
    print(value_loss)

    loss.backward()
    optimizer.step()

    if episode % 1 == 0:
        print(f"Episode {episode}, Loss: {loss.item()}")
        # print(loss)
        # print(state.board)  # 查看盘面
        print(final_reward)
    # break

tensor(4.1580, grad_fn=<NllLossBackward0>)

tensor(1.0002, grad_fn=<MeanBackward0>)

Episode 0, Loss: 5.158205509185791

1

tensor(4.1580, grad_fn=<NllLossBackward0>)

...

-1

tensor(4.1149, grad_fn=<NllLossBackward0>)

tensor(1.0001, grad_fn=<MeanBackward0>)

Episode 199, Loss: 5.114990234375

-1

## MCTS Inference

In [11]:
state = GomokuState(board_size=board_size, gomoku_number=gomoku_number)
while not state.is_terminal():
    action = mcts_move(state, net, 100)
    state.move(action)
    print(state.board)
print(state.get_reward())
print("Game over")

tensor([[[[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 1., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]]]])

tensor([[[[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [-1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]]]])

tensor([[[[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [-1.,  0.,  0.,  1.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]]]])

tensor([[[[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [-1.,  0.,  0.,  1.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0., -1.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]]]])

tensor([[[[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [-1.,  0.,  0.,  1.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0., -1.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]]]])

...

tensor([[[[-1.,  0., -1.,  1.,  0.,  1.,  1.,  0.],
          [ 1., -1., -1.,  0., -1.,  1., -1.,  1.],
          [-1.,  1.,  1.,  1., -1.,  0.,  0., -1.],
          [ 0.,  0., -1., -1.,  0.,  1.,  1., -1.],
          [-1., -1.,  0.,  0.,  1., -1.,  0.,  1.],
          [ 0.,  0.,  1., -1., -1.,  1., -1.,  1.],
          [ 1., -1., -1., -1.,  1.,  1., -1., -1.],
          [ 1.,  1.,  1., -1.,  1., -1.,  1.,  1.]]]])

tensor([[[[-1.,  0., -1.,  1.,  0.,  1.,  1.,  0.],
          [ 1., -1., -1.,  0., -1.,  1., -1.,  1.],
          [-1.,  1.,  1.,  1., -1.,  0.,  0., -1.],
          [ 0.,  0., -1., -1.,  0.,  1.,  1., -1.],
          [-1., -1.,  0.,  0.,  1., -1.,  0.,  1.],
          [ 0.,  1.,  1., -1., -1.,  1., -1.,  1.],
          [ 1., -1., -1., -1.,  1.,  1., -1., -1.],
          [ 1.,  1.,  1., -1.,  1., -1.,  1.,  1.]]]])

tensor([[[[-1.,  0., -1.,  1.,  0.,  1.,  1.,  0.],
          [ 1., -1., -1.,  0., -1.,  1., -1.,  1.],
          [-1.,  1.,  1.,  1., -1.,  0.,  0., -1.],
          [ 0.,  0., -1., -1.,  0.,  1.,  1., -1.],
          [-1., -1.,  0., -1.,  1., -1.,  0.,  1.],
          [ 0.,  1.,  1., -1., -1.,  1., -1.,  1.],
          [ 1., -1., -1., -1.,  1.,  1., -1., -1.],
          [ 1.,  1.,  1., -1.,  1., -1.,  1.,  1.]]]])

-1

Game over

## MCTS与LLM关系

1. 区别于cartpole，这里的agent在下子时有policy network和value network。
2. 这里要求value估计接近树回溯值
3. 棋子的状态可以看成是连续的(2d棋盘有-1,0,1)，棋子的动作看成是有限的离散集合（动作范围15*15）。
4. LLM的状态是连续的。动作是离散的(词表大小）。这里的问题在于搜索空间更大如llama3为128k
5. LLM与go之间的差异在于，以逐个token来采集，树木深度高，如1024深度，模拟采样的成本过高，且高效采样到terminal成功的难度大，导致有效feedback太少。
6. 如何减少模拟采样的成本，如何有效的采样的正确的推理step，如何得到准确的feedback，是LLM做MCTS-like搜索的关键。
7. 针对6如何来解决？ 

reference：claude-3.5-sonnet